# InkyNano Full Expert GGUF Conversion (Google Colab, Resumable CPU-First)

Converts all **3,072 physical experts** (`24 layers x 128 experts`) of `thinkingmachines/Inkling-2-Mini` from native **NVFP4** (layers 0–19) and **MXFP8** (layers 20–23) into individually addressable **GGML-quantized GGUF v3 tensor bundles**, plus a separately quantized **text-only backbone** preserving sensitive routing and recurrent control tensors at native BF16/F32 precision.

### Key Architectural Contracts
1. **Verified Native Dequantization:** Reuses the exact `Inkling2LoopDequantizeExperts` + `Interleave(dim=1)` (`loader_cast=True`) path verified to `0.0 max_abs` in `SilicoExperiments` PR #3 (`transformers@d8f10a82213a3a9db48385aba3a4b57ab535de50`).
2. **Actual GGML Quantization Routines:** Compiles pinned upstream `ggml-org/llama.cpp` (`libggml-base.so` + `libggml-cpu.so`) on CPU and invokes `ggml_quantize_chunk` and `ggml_get_type_traits->to_float` directly.
3. **Standalone Tensor Bundles vs Complete `llama.cpp` Model:** Produced `.gguf` files set `general.type = "tensors"` and `inkynano.llama_cpp_dropin_compatible = false`. They are **NOT** a drop-in `llama-cli` model (upstream `llama.cpp` does not implement Inkling-2-Loop's 3-pass recurrent core, block attention residuals, short convolutions, or soft-capped SwiGLU `gate_cap=4.0`, `up_cap=25.0`). They are engineered for **selective per-expert byte-offset loading and LDT-controlled residency on RTX 3050 (4 GiB)**.
4. **Bounded Memory & Disconnect-Safe Resume:** Streams one physical expert (or one 4,096-row backbone stripe) at a time directly to disk via 2-pass `StreamingGGUFWriter`, uploads each completed layer shard + receipt atomically to private `AlephFunk/InkyNano-GGUF`, deletes the local shard after verified upload, and resumes cleanly after any Colab disconnect without repeating completed shards.
5. **CPU-First Runtime:** Start on a standard or High-RAM **CPU** runtime. Only request a GPU runtime if the 16-expert benchmark profiling cell reports `gpu_recommended: True`.

In [ ]:
import os, platform, shutil, sys

print("Python:", sys.version)
print("Platform:", platform.platform())
print("CPU cores:", os.cpu_count())

try:
    with open("/proc/meminfo") as f:
        mem = {line.split(":")[0]: int(line.split()[1]) // 1024 for line in f if line.startswith(("MemTotal:", "MemAvailable:"))}
    print(f"RAM: total={mem.get('MemTotal', 0) / 1024:.2f} GiB, available={mem.get('MemAvailable', 0) / 1024:.2f} GiB")
except FileNotFoundError:
    pass

du = shutil.disk_usage("/content" if os.path.exists("/content") else ".")
print(f"Disk: total={du.total / (1<<30):.1f} GiB, free={du.free / (1<<30):.1f} GiB")

try:
    import torch
    print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("NOTE: GPU runtime detected. CPU runtime is preferred unless benchmark profiling recommends GPU.")
except ImportError:
    print("PyTorch will be installed in Cell 3.")

In [ ]:
import os
from pathlib import Path

# --- Repository & Conversion Configuration ---
REPO_URL = "https://github.com/MoralityLabAI/LoopedTransformers.git"
REPO_BRANCH = "research/inky-nano-prep"

HF_REPO_ID = "AlephFunk/InkyNano-GGUF"   # Private destination repository
PRIVATE_HF_REPO = True

# Expert profile chosen AFTER reviewing Cell 5 (16-expert benchmark):
#   "pure_q4_k"       -> all 24 layers: gate/up=Q4_K, down=Q5_0 (canonical llama.cpp fallback for ne0=1152)
#   "mixed_q4_k_q6_k" -> layers 0-19 (NVFP4): Q4_K/Q4_K/Q5_0; layers 20-23 (MXFP8): Q6_K/Q6_K/Q8_0
EXPERT_PROFILE = "mixed_q4_k_q6_k"
BACKBONE_PROFILE = "backbone_mixed_q4_k_q6_k"

# "local": download pinned 19.5 GB shard once to Colab local SSD (/content/inky_source) then stream 1 expert at a time.
# "remote": zero full-file download; stream each expert via HTTP 206 byte-range requests (slower for all 3,072 experts).
SOURCE_MODE = "local"
KEEP_LOCAL_SHARDS = False     # Delete local shard after verified HF upload to keep Colab disk bounded
USE_GOOGLE_DRIVE = False      # Optional: persist receipts/state on Drive in addition to HF remote resume
MAX_RSS_GIB = 8.0             # Host memory guard ceiling
N_THREADS = min(os.cpu_count() or 2, 8)
LAYERS_SPEC = "0-23"          # Override e.g. "0-5" for partial runs

BASE_DIR = Path("/content/inkynano_gguf_work")
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/InkyNano-GGUF-Work")
else:
    WORK_DIR = BASE_DIR / "out"

REPO_DIR = BASE_DIR / "LoopedTransformers"
GGML_DIR = BASE_DIR / "ggml_build"
SOURCE_DIR = BASE_DIR / "source_snapshot"
WORK_DIR.mkdir(parents=True, exist_ok=True)

# --- Secure Hugging Face Token Lookup (never hardcoded or printed) ---
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
        if tok:
            os.environ["HF_TOKEN"] = tok.strip()
    except Exception:
        pass
if not os.environ.get("HF_TOKEN"):
    import getpass
    os.environ["HF_TOKEN"] = getpass.getpass("Enter Hugging Face write token for AlephFunk/InkyNano-GGUF: ").strip()

print("Config ready:")
print("  WORK_DIR       :", WORK_DIR)
print("  HF_REPO_ID     :", HF_REPO_ID, f"(private={PRIVATE_HF_REPO})")
print("  EXPERT_PROFILE :", EXPERT_PROFILE)
print("  BACKBONE       :", BACKBONE_PROFILE)
print("  SOURCE_MODE    :", SOURCE_MODE)
print("  N_THREADS      :", N_THREADS)
print("  HF_TOKEN set   :", bool(os.environ.get("HF_TOKEN")))

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

TRANSFORMERS_REV = "d8f10a82213a3a9db48385aba3a4b57ab535de50"
LLAMA_CPP_COMMIT = "ae6a7ad2c5754dd8db93d04c84fda70435864f6e"

# 1. Clone or update LoopedTransformers PR #341 branch
if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--single-branch", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)

repo_sha = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("LoopedTransformers commit:", repo_sha)

# 2. Install pinned runtime dependencies without disturbing Colab PyTorch
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "safetensors>=0.8.0", "huggingface_hub>=0.29.0", "tokenizers>=0.21.0"],
    check=True,
)

# Install exact upstream transformers commit verified in SilicoExperiments PR #3
tf_src = BASE_DIR / "transformers_src"
need_tf = True
try:
    import transformers
    from transformers.core_model_loading import Inkling2LoopDequantizeExperts, Interleave  # noqa: F401
    print("Pinned Inkling2LoopDequantizeExperts already importable; transformers", transformers.__version__)
    need_tf = False
except Exception:
    pass

if need_tf:
    if not (tf_src / ".git").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--filter=blob:none", "https://github.com/huggingface/transformers.git", str(tf_src)],
            check=True,
        )
    subprocess.run(["git", "-C", str(tf_src), "fetch", "--depth", "1", "origin", TRANSFORMERS_REV], check=True)
    subprocess.run(["git", "-C", str(tf_src), "checkout", "--detach", "FETCH_HEAD"], check=True)
    got_tf = subprocess.check_output(["git", "-C", str(tf_src), "rev-parse", "HEAD"], text=True).strip()
    assert got_tf == TRANSFORMERS_REV, f"Wrong transformers commit: {got_tf}"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(tf_src)], check=True)
    import importlib
    import transformers
    importlib.reload(transformers)
    from transformers.core_model_loading import Inkling2LoopDequantizeExperts, Interleave  # noqa: F401
    print("Installed pinned transformers:", transformers.__version__, "@", got_tf[:12])

# 3. Build actual upstream GGML C quantization library (CPU-only, ~30s)
inkynano_dir = str(REPO_DIR / "InkyNano")
if inkynano_dir not in sys.path:
    sys.path.insert(0, inkynano_dir)

from gguf_backend import build_ggml_cpu
ggml_receipt = build_ggml_cpu(GGML_DIR, commit=LLAMA_CPP_COMMIT, jobs=N_THREADS)
os.environ["INKYNANO_GGML_LIB_DIR"] = ggml_receipt["lib_dir"]
print("GGML C backend ready:", json.dumps(ggml_receipt, indent=2))

In [ ]:
import os, subprocess, sys

# 1. Bounded remote metadata inventory (no weights downloaded)
inv_path = WORK_DIR / "inventory.json"
subprocess.run(
    [sys.executable, str(REPO_DIR / "InkyNano" / "inventory.py"), "--remote", "--output", str(inv_path)],
    check=True,
)

# 2. Run full InkyNano unit test suite with live GGML C backend enabled
env = {**os.environ, "INKYNANO_GGML_LIB_DIR": os.environ["INKYNANO_GGML_LIB_DIR"]}
subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", str(REPO_DIR / "InkyNano"), "-p", "test_*.py", "-v"],
    env=env,
    check=True,
)
print("Preflight inventory and all unit tests PASSED.")

In [ ]:
import json
from gguf_convert import BENCHMARK_RECEIPT_RELPATH, get_backend, run_benchmark_16
from gguf_dequant import CHECKPOINT_SHARD, open_source

# Stream ONLY the 16 stratified experts (layers 0, 6, 17, 23 x 4, seed 17) over HTTP Range
# (~82 MB total) unless the local checkpoint is already present.
local_ckpt = SOURCE_DIR / CHECKPOINT_SHARD
src_arg = local_ckpt if local_ckpt.is_file() else None

backend = get_backend()
with open_source(src_arg) as source:
    bench = run_benchmark_16(
        source=source,
        backend=backend,
        out_dir=WORK_DIR,
        dequant_backend="transformers",
        n_threads=N_THREADS,
        max_rss_gib=MAX_RSS_GIB,
    )

print("=== Transformers vs Standalone Native Dequant Parity ===")
for chk in bench["parity_checks_transformers_vs_standalone"]:
    print(f"  L{chk['layer']:02d}/E{chk['expert']:03d} ({chk['native_format']}): "
          f"w13 max_abs={chk['w13_transformers_vs_standalone']['max_abs']}, "
          f"w2 max_abs={chk['w2_transformers_vs_standalone']['max_abs']}")

print("\n=== 16-Expert Comparison: Pure Q4_K vs Mixed Q4_K/Q6_K ===")
header = f"{'Profile':26s} | {'NVFP4':16s} | {'MXFP8':16s} | {'Bank GiB':8s} | {'SwiGLU L2':9s} | {'Heavy L2':9s} | {'Mix L2':8s} | {'Proj min':8s}"
print(header)
print("-" * len(header))
for pname, s in bench["profile_summaries"].items():
    nvt = "/".join(s["nvfp4_quant_types"][k] for k in ("gate", "up", "down"))
    mxt = "/".join(s["mxfp8_quant_types"][k] for k in ("gate", "up", "down"))
    print(
        f"{pname:26s} | {nvt:16s} | {mxt:16s} | {s['full_bank_payload_gib']:8.3f} | "
        f"{s['swiglu_unit_rms_rel_l2_mean']:9.5f} | {s['swiglu_heavy_tail_rel_l2_mean']:9.5f} | "
        f"{s['mixture_2expert_rel_l2_mean']:8.5f} | {s['projected_3072_total_minutes']:8.2f}"
    )

print("\n=== CPU vs GPU Profiling Gate ===")
print(json.dumps(bench["cpu_vs_gpu_profiling"], indent=2))
print("\nSaved receipt:", WORK_DIR / BENCHMARK_RECEIPT_RELPATH)

In [ ]:
from huggingface_hub import snapshot_download
from gguf_dequant import CHECKPOINT_SHARD, CHECKPOINT_SIZE, MODEL, REV

# Always download tiny metadata/tokenizer files for self-contained HF bundle packaging
meta_patterns = [
    "config.json",
    "hf_quant_config.json",
    "model.safetensors.index.json",
    "tokenizer*",
    "special_tokens_map.json",
    "generation_config.json",
    "chat_template*",
]
patterns = list(meta_patterns)
if SOURCE_MODE == "local":
    patterns.append(CHECKPOINT_SHARD)

SOURCE_DIR.mkdir(parents=True, exist_ok=True)
snapshot_download(
    repo_id=MODEL,
    revision=REV,
    local_dir=str(SOURCE_DIR),
    allow_patterns=patterns,
    max_workers=4,
)

if SOURCE_MODE == "local":
    weight_file = SOURCE_DIR / CHECKPOINT_SHARD
    actual = weight_file.stat().st_size
    assert actual == CHECKPOINT_SIZE, f"Unexpected checkpoint size: {actual} != {CHECKPOINT_SIZE}"
    print(f"Verified pinned local checkpoint: {weight_file} ({actual / (1<<30):.3f} GiB)")
else:
    print("SOURCE_MODE='remote': streaming experts directly over HTTP 206 Range requests.")

In [ ]:
import json
from gguf_convert import get_backend, parse_layer_spec, run_all_resumable
from gguf_dequant import CHECKPOINT_SHARD, open_source

src_path = (SOURCE_DIR / CHECKPOINT_SHARD) if SOURCE_MODE == "local" else None
backend = get_backend()

with open_source(src_path) as source:
    state = run_all_resumable(
        source=source,
        backend=backend,
        out_dir=WORK_DIR,
        profile_name=EXPERT_PROFILE,
        backbone_profile=BACKBONE_PROFILE,
        layers=parse_layer_spec(LAYERS_SPEC),
        hf_repo=HF_REPO_ID,
        private_repo=PRIVATE_HF_REPO,
        keep_local_shards=KEEP_LOCAL_SHARDS,
        dequant_backend="transformers",
        n_threads=N_THREADS,
        max_rss_gib=MAX_RSS_GIB,
        source_config_dir=SOURCE_DIR,
    )

print("\n=== Conversion & Incremental Upload Summary ===")
print(json.dumps(state, indent=2))

In [ ]:
import json, numpy as np, torch
from huggingface_hub import hf_hub_download
from gguf_convert import MANIFEST_RELPATH, get_backend, parse_layer_spec, run_all_resumable
from gguf_dequant import CHECKPOINT_SHARD, open_source
from gguf_loader import LDTResidencyManager, SelectiveExpertStore

# 1. Resume idempotency check: re-running must skip every completed shard and backbone
src_path = (SOURCE_DIR / CHECKPOINT_SHARD) if SOURCE_MODE == "local" else None
backend = get_backend()
with open_source(src_path) as source:
    resume_state = run_all_resumable(
        source=source,
        backend=backend,
        out_dir=WORK_DIR,
        profile_name=EXPERT_PROFILE,
        backbone_profile=BACKBONE_PROFILE,
        layers=parse_layer_spec(LAYERS_SPEC),
        hf_repo=HF_REPO_ID,
        private_repo=PRIVATE_HF_REPO,
        keep_local_shards=KEEP_LOCAL_SHARDS,
        dequant_backend="transformers",
        n_threads=N_THREADS,
        max_rss_gib=MAX_RSS_GIB,
        source_config_dir=SOURCE_DIR,
    )

assert len(resume_state["converted_layers"]) == 0, f"Resume re-converted layers: {resume_state['converted_layers']}"
assert resume_state["backbone_status"].startswith("skipped"), f"Resume re-converted backbone: {resume_state['backbone_status']}"
print("Resume idempotency check PASSED: 0 layers and 0 backbones re-converted.")

# 2. Verify manifest completeness (3,072 experts across 24 shards + text-only backbone)
manifest_path = WORK_DIR / MANIFEST_RELPATH
manifest = json.loads(manifest_path.read_text(encoding="utf8"))
assert manifest["llama_cpp_dropin_compatible"] is False
assert manifest["num_layers"] == len(parse_layer_spec(LAYERS_SPEC))
assert manifest["total_experts"] == 128 * len(parse_layer_spec(LAYERS_SPEC))
assert manifest["backbone"] is not None
print(
    f"Manifest verified: {manifest['num_layers']} shards, {manifest['total_experts']} experts, "
    f"bank={manifest['total_shard_bytes'] / (1<<30):.3f} GiB, "
    f"backbone={manifest['backbone']['bytes'] / (1<<30):.3f} GiB"
)

# 3. Spot-check selective per-expert byte-offset loading + LDT residency on Layer 0
verify_root = BASE_DIR / "verify_spot"
verify_root.mkdir(parents=True, exist_ok=True)
shard0_local = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename="shards/inkynano_experts_layer_00.gguf",
    local_dir=str(verify_root),
    token=os.environ.get("HF_TOKEN"),
)
store = SelectiveExpertStore(manifest_path, root_dir=verify_root, backend=backend)
ldt = LDTResidencyManager(store, capacity_per_layer=8, verify_sha256=True)
scores = np.linspace(1.0, 0.0, 128, dtype=np.float64)
admit_info = ldt.admit_layer(layer=0, scores=scores, swap_budget=8)
x = torch.randn(4, 2304, dtype=torch.bfloat16)
y = ldt.forward_resident_mixture(layer=0, x=x, selected_experts=[0, 1], weights=[0.37, 0.63])
assert torch.isfinite(y).all() and y.shape == (4, 2304)
print("Selective loader & LDT residency spot-check PASSED:", admit_info, "output_norm=", float(y.float().norm()))